# 🤖 Thực Hành: Lập Trình Scaled Dot-Product & Multi-Head Attention Từ Con Số Không

Chào Khang! Trong bài tập thực hành này, bạn sẽ trực tiếp sử dụng thư viện **PyTorch** để lập trình từ con số không hai thành phần quan trọng nhất tạo nên sức mạnh cho mô hình Transformer:
1. Cơ chế **Scaled Dot-Product Attention**.
2. Lớp **Multi-Head Attention** hoàn chỉnh.

Việc tự tay code giúp bạn làm quen với các thao tác chuyển đổi chiều ma trận (tensor manipulation) kinh điển trong Deep Learning như `.transpose()`, `.view()`, và phép nhân ma trận đa chiều `torch.matmul()`.

In [ ]:
# Cài đặt PyTorch và các thư viện vẽ đồ thị bổ trợ
!pip install torch numpy matplotlib

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

# Thiết lập seed để kết quả đồng nhất giữa các lần chạy
torch.manual_seed(42)
print("Đã tải thành công PyTorch!")

---
## 1. Lập Trình Cơ Chế Scaled Dot-Product Attention

Công thức cốt lõi:
$$\text{Attention}(Q, K, V) = \text{Softmax}\left(\frac{Q K^T}{\sqrt{d_k}}\right) V$$

**Nhiệm vụ của bạn:** Hoàn thành hàm `scaled_dot_product_attention` dưới đây.

*Lưu ý:* 
- Ma trận Q có kích thước `(batch_size, n_heads, seq_len, d_k)`.
- Để nhân ma trận Q với ma trận chuyển vị của K ở hai chiều cuối cùng, ta dùng: `k.transpose(-2, -1)`.
- Sử dụng `torch.matmul(A, B)` để nhân hai ma trận.
- Sử dụng `F.softmax(..., dim=-1)`.

In [ ]:
def scaled_dot_product_attention(q, k, v, mask=None):
    """
    Tính toán cơ chế Scaled Dot-Product Attention.
    q: Query tensor, kích thước (batch_size, n_heads, seq_len, d_k)
    k: Key tensor, kích thước (batch_size, n_heads, seq_len, d_k)
    v: Value tensor, kích thước (batch_size, n_heads, seq_len, d_v)
    mask: Tensor che (tùy chọn) để chặn attention vào một số phần tử đầu vào.
    """
    # Lấy số chiều đặc trưng d_k từ tensor q
    d_k = q.size(-1)
    
    # BƯỚC 1: Tính toán scores = (Q * K^T) / sqrt(d_k)
    # ------------------ YOUR CODE HERE ------------------
    scores = None
    # ----------------------------------------------------
    
    # Áp dụng mask nếu có (ví dụ trong mô hình sinh từ tuần tự, che đi các từ tương lai)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)
        
    # BƯỚC 2: Đưa scores qua hàm Softmax ở chiều cuối cùng (dim=-1) để nhận trọng số attention
    # ------------------ YOUR CODE HERE ------------------
    attention_weights = None
    # ----------------------------------------------------
    
    # BƯỚC 3: Nhân trọng số attention_weights với ma trận Value (V)
    # ------------------ YOUR CODE HERE ------------------
    output = None
    # ----------------------------------------------------
    
    return output, attention_weights

In [ ]:
# --- BỘ TEST KIỂM THỬ TỰ ĐỘNG HÀM ATTENTION ---
bs, heads, seq, d = 2, 8, 5, 64
q_test = torch.randn(bs, heads, seq, d)
k_test = torch.randn(bs, heads, seq, d)
v_test = torch.randn(bs, heads, seq, d)

try:
    out, weights = scaled_dot_product_attention(q_test, k_test, v_test)
    assert out.shape == (bs, heads, seq, d), f"Sai kích thước output: {out.shape}"
    assert weights.shape == (bs, heads, seq, seq), f"Sai kích thước weights: {weights.shape}"
    assert torch.allclose(weights.sum(dim=-1), torch.ones(bs, heads, seq)), "Tổng trọng số Softmax phải bằng 1.0!"
    print("🎉 Tuyệt vời! Bộ test tự động đã vượt qua thành công! Hàm attention viết cực chuẩn.")
except Exception as e:
    print("❌ Lỗi rồi Khang ơi! Kiểm tra lại code nhân ma trận hoặc chia tỷ lệ nhé:")
    print(e)

---
## 2. Trực Quan Hóa Ma Trận Attention (Attention Map)

Hãy xem cách một từ tập trung chú ý vào các từ khác thông qua biểu đồ nhiệt **Heatmap**.

In [ ]:
# Khởi tạo Query & Key đặc biệt để tạo ra sự chú ý lệch tập trung
q_demo = torch.zeros(1, 1, 6, 8)
k_demo = torch.zeros(1, 1, 6, 8)

# Đặt độ tương đồng lớn giữa từ 2 và từ 4, từ 5 và từ 1
q_demo[0, 0, 2, :] = torch.tensor([1.5, 0.0, 2.0, 0.0, 1.0, 0.0, 0.0, 0.0])
k_demo[0, 0, 4, :] = torch.tensor([1.5, 0.0, 2.0, 0.0, 1.0, 0.0, 0.0, 0.0])
v_demo = torch.randn(1, 1, 6, 8)

_, weights = scaled_dot_product_attention(q_demo, k_demo, v_demo)
weights_np = weights.squeeze().detach().numpy()

# Vẽ đồ thị Heatmap biểu thị độ mạnh chú ý
words = ["The", "animal", "didn't", "cross", "the", "street"]
plt.figure(figsize=(8, 6))
plt.imshow(weights_np, cmap='plasma')
plt.colorbar(label='Độ mạnh của Attention (Softmax score)')
plt.xticks(range(len(words)), words, rotation=45)
plt.yticks(range(len(words)), words)
plt.title("Biểu đồ nhiệt Attention Heatmap Trực Quan")
plt.show()

---
## 3. Lập Trình Class Multi-Head Attention Hoàn Chỉnh

Multi-Head Attention chiếu dữ liệu đầu vào thành nhiều đầu nhỏ song song để học đa chiều ngữ nghĩa.

**Nhiệm vụ của bạn:** Điền mã nguồn vào phương thức `split_heads` (phân mảnh ma trận thành các head) và phương thức `forward`.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super(MultiHeadAttention, self).__init__()
        assert d_model % n_heads == 0, "d_model phải chia hết cho n_heads!"
        
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        
        # Định nghĩa các lớp Linear chiếu tuyến tính cho Q, K, V
        self.q_linear = nn.Linear(d_model, d_model)
        self.k_linear = nn.Linear(d_model, d_model)
        self.v_linear = nn.Linear(d_model, d_model)
        
        # Lớp Linear chiếu tuyến tính đầu ra gộp
        self.out_linear = nn.Linear(d_model, d_model)
        
    def split_heads(self, x, batch_size):
        """
        Thay đổi kích thước của x từ (batch_size, seq_len, d_model)
        thành (batch_size, n_heads, seq_len, d_k) để chạy song song.
        """
        # ------------------ YOUR CODE HERE ------------------
        # Bước 1: Reshape x sang (batch_size, seq_len, n_heads, d_k) bằng .view()
        # Bước 2: Chuyển vị trục sang (batch_size, n_heads, seq_len, d_k) bằng .transpose()
        x_split = None
        # ----------------------------------------------------
        return x_split
        
    def forward(self, q, k, v, mask=None):
        batch_size = q.size(0)
        
        # 1. Chiếu tuyến tính đầu vào qua các linear layers
        q = self.q_linear(q)
        k = self.k_linear(k)
        v = self.v_linear(v)
        
        # 2. Phân mảnh ma trận thành các heads độc lập
        q = self.split_heads(q, batch_size)
        k = self.split_heads(k, batch_size)
        v = self.split_heads(v, batch_size)
        
        # 3. Tính Scaled Dot-Product Attention song song trên tất cả các heads
        # ------------------ YOUR CODE HERE ------------------
        attn_output, attn_weights = None, None
        # ----------------------------------------------------
        
        # 4. Gộp các heads lại bằng cách đảo ngược quá trình split
        # Kích thước chuyển từ (batch_size, n_heads, seq_len, d_k)
        # về (batch_size, seq_len, n_heads, d_k) rồi dùng .contiguous().view() về (batch_size, seq_len, d_model)
        attn_output = attn_output.transpose(1, 2).contiguous()
        concat_output = attn_output.view(batch_size, -1, self.d_model)
        
        # 5. Chiếu tuyến tính đầu ra gộp qua out_linear
        output = self.out_linear(concat_output)
        
        return output, attn_weights

In [ ]:
# --- TEST THỬ MÔ HÌNH MULTI-HEAD ATTENTION ---
mha = MultiHeadAttention(d_model=128, n_heads=8)
x_input = torch.randn(4, 10, 128)  # batch=4, seq_len=10, features=128

try:
    out, weights = mha(x_input, x_input, x_input)
    print("--- THÔNG SỐ ĐẦU RA MULTI-HEAD ATTENTION ---")
    print("Kích thước x_input ban đầu:", x_input.shape)
    print("Kích thước output gộp:     ", out.shape)
    print("Kích thước attention map:   ", weights.shape)
    
    assert out.shape == x_input.shape, "Đầu ra của Multi-Head Attention phải trùng khớp chiều với đầu vào!"
    print("\n🎉 Chúc mừng Khang! Lớp Multi-Head Attention hoạt động cực kỳ hoàn hảo! Bạn đã nắm trọn vẹn mô hình rồi!")
except Exception as e:
    print("❌ Lỗi rồi Khang ơi! Kiểm tra lại phần reshape chia đầu split_heads hoặc chuyển vị nhé:")
    print(e)

---
## 4. Câu Hỏi Đào Sâu Suy Nghĩ Dành Cho Khang

1. **Bản chất của Positional Encoding**: Tại sao mô hình Transformer bắt buộc phải cộng thêm vector Positional Encoding vào các vector từ nhúng đầu vào? RNN hay LSTM có cần Positional Encoding không? Tại sao?
2. **Phân tích Linformer**: Khi ta giảm chiều độ phức tạp bằng Linformer (chiếu từ $T \times T$ sang $T \times k$), điều này có làm mô hình mất mát thông tin ngữ cảnh nào không? Trong trường hợp nào phép chiếu này sẽ hoạt động kém hiệu quả?